## 1. Setup & Imports

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers pymupdf accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 15.5 MB/s eta 0:00:00


In [6]:
import fitz
import numpy as np
import faiss
import torch
import json
from datetime import datetime
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## 2. Extracting raw Text from PDF


In [7]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text = ""
    with fitz.open(pdf_path) as pdf:
        for page_num in range(pdf.page_count):
            page = pdf.load_page(page_num)
            text += page.get_text("text")
    return text

In [8]:
pdf_path = "/content/sample.pdf"
raw_text = extract_text_from_pdf(pdf_path)
print(len(raw_text))
print(raw_text[:500])

36894
“Virtualization in Cloud 
Computing”  
Prepared By 
Prof. Anand N. Gharu  
(Assistant Professor) 
Computer Engineering Departement 
25 May 2022 
. 
CLASS      : TE COMPUTER 2019 
SUBJECT : CC (SEM-II) 
UNIT         : III 
1 
SYLLABUS 
Introduction: Definition of Virtualization, Adopting Virtualization, 
Types of Virtualization, Virtualization Architecture and Software, 
Virtual 
Clustering, 
Virtualization 
Application, 
Pitfalls 
of 
Virtualization. Grid, Cloud and Virtualization: Virtualizatio


### 3.Chunking

In [9]:
def split_text(text: str, chunk_size: int = 1000, overlap: int = 200) -> list:
    sentences = text.split(". ")
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk)
            raw_overlap = current_chunk[-overlap:] if len(current_chunk) > overlap else current_chunk
            first_space = raw_overlap.find(" ")
            overlap_text = raw_overlap[first_space + 1:] if first_space != -1 else raw_overlap
            current_chunk = overlap_text + sentence + ". "
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

In [10]:
chunks = split_text(raw_text, chunk_size=1000, overlap=200)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 52


### 4. Embedding

In [11]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)
print(chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(52, 384)


### 5. Vector Database

In [12]:
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings).astype('float32'))

print(f"Vectors in index: {index.ntotal}")

Vectors in index: 52


### 6. Re-ranking


In [13]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

### 7. Context Retrieval

In [14]:
def retrieve(query: str, initial_k: int = 10, final_k: int = 3) -> list:
    query_embedding = embedder.encode([query]).astype('float32')
    _, indices = index.search(query_embedding, initial_k)
    candidate_chunks = [chunks[i] for i in indices[0]]

    pairs = [[query, chunk] for chunk in candidate_chunks]
    scores = reranker.predict(pairs)

    reranked = sorted(zip(candidate_chunks, scores), key=lambda x: x[1], reverse=True)
    return [chunk for chunk, score in reranked[:final_k]]

In [15]:
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### 8. Answer Generation

In [18]:
def generate_answer(question: str, initial_k: int = 10, final_k: int = 3, max_new_tokens: int = 150, max_context_tokens: int = 350) -> str:
    retrieved_chunks = retrieve(question, initial_k=initial_k, final_k=final_k)
    context = "\n\n".join(retrieved_chunks)

    context_tokens = tokenizer(context, truncation=True, max_length=max_context_tokens)["input_ids"]
    context = tokenizer.decode(context_tokens, skip_special_tokens=True)

    prompt = (
        f"Using only the information in the context, write a complete, clear answer to the question. "
        f"If the question asks for multiple items or a comparison, cover all relevant points from the context, not just one.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### Testing

In [20]:
question = "What is virtualization?"
answer = generate_answer(question)
print(f"Q: {question}")
print(f"A: {answer}")

Q: What is virtualization?
A: A technique, which allows to share a single physical instance of a resource or an application among multiple customers and organizations.


### 9. Validation Logs

In [21]:
validation_questions = [
    "What is a hypervisor?",
    "What is the difference between a virtual machine and a container?",
    "What are the types of virtualization?",
    "What is server virtualization used for?",
    "What is storage virtualization?",
]

validation_log = []
for q in validation_questions:
    retrieved_chunks = retrieve(q)
    answer = generate_answer(q)
    validation_log.append({
        "timestamp": datetime.now().isoformat(),
        "question": q,
        "retrieved_chunks_preview": [c[:150] for c in retrieved_chunks],
        "generated_answer": answer,
    })

with open("validation_log.json", "w") as f:
    json.dump(validation_log, f, indent=2)

for entry in validation_log:
    print(f"Q: {entry['question']}")
    print(f"A: {entry['generated_answer']}")
    print(f"Top retrieved chunk: {entry['retrieved_chunks_preview'][0]}...")
    print("-" * 60)

print("\nSaved to validation_log.json")

Q: What is a hypervisor?
A: The hypervisor isolates operating systems and applications from the underlying computer hardware so the host machine can run multiple virtual machines (VM) as guests that share the system's physical compute resources, such as processor cycles, memory space, network bandwidth and so on.
Top retrieved chunk: of the particular components 
involved in delivering a virtual -- rather than physical -- version of 
something, such as an operating system (OS), a s...
------------------------------------------------------------
Q: What is the difference between a virtual machine and a container?
A: They are a collection of processes that executes along with corresponding namespace or identifiers of process.
Top retrieved chunk: the operating system 
that runs on actual hardware. A virtual counterpart of the 
operating system is a subpart that executes or emulates the 
virtual...
------------------------------------------------------------
Q: What are the types of virtu

## 10. System Metrics Report

In [22]:
system_report = {
    "document_ingestion": {
        "tool": "PyMuPDF (fitz)",
        "raw_text_length_chars": len(raw_text),
    },
    "chunking": {
        "strategy": "sentence-accumulation with word-boundary-safe overlap",
        "chunk_size": 1000,
        "overlap": 200,
        "num_chunks": len(chunks),
    },
    "embedding_model": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "dimensions": chunk_embeddings.shape[1],
        "num_vectors": chunk_embeddings.shape[0],
    },
    "vector_store": {
        "tool": "FAISS",
        "index_type": "IndexFlatL2 (exact search)",
        "vectors_indexed": index.ntotal,
    },
    "retrieval_strategy": {
        "method": "Two-stage retrieval: FAISS vector search (top 10 candidates) + cross-encoder re-ranking (top 3 final)",
        "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "note": "A BM25+vector hybrid (Reciprocal Rank Fusion) was also implemented and tested; cross-encoder re-ranking was kept as final since it gave more consistent generation results across test questions.",
    },
    "language_model": {
        "name": "google/flan-t5-large",
        "parameters": "~780M",
        "task": "text2text-generation (seq2seq), loaded via AutoModelForSeq2SeqLM",
        "max_context_tokens": 350,
        "max_new_tokens": 150,
    },
}

with open("system_metrics_report.json", "w") as f:
    json.dump(system_report, f, indent=2)

print(json.dumps(system_report, indent=2))

{
  "document_ingestion": {
    "tool": "PyMuPDF (fitz)",
    "raw_text_length_chars": 36894
  },
  "chunking": {
    "strategy": "sentence-accumulation with word-boundary-safe overlap",
    "chunk_size": 1000,
    "overlap": 200,
    "num_chunks": 52
  },
  "embedding_model": {
    "name": "sentence-transformers/all-MiniLM-L6-v2",
    "dimensions": 384,
    "num_vectors": 52
  },
  "vector_store": {
    "tool": "FAISS",
    "index_type": "IndexFlatL2 (exact search)",
    "vectors_indexed": 52
  },
  "retrieval_strategy": {
    "method": "Two-stage retrieval: FAISS vector search (top 10 candidates) + cross-encoder re-ranking (top 3 final)",
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "note": "A BM25+vector hybrid (Reciprocal Rank Fusion) was also implemented and tested; cross-encoder re-ranking was kept as final since it gave more consistent generation results across test questions."
  },
  "language_model": {
    "name": "google/flan-t5-large",
    "parameters": 